In [ ]:
import numpy as np
import matplotlib as plt
from pathlib import Path
import pandas as pd
from scipy.stats import vonmises
from scipy.special import i0
from math import sin
from math import cos


# run functions

### def wrap_angle_rad

功能：把任意弧度角限制到 $[-\pi, \pi)$，用于计算圆周角度差。

输入：`theta`，标量或数组形式的弧度角。

输出：与输入形状相同的圆周范围内角度。


In [3]:
def wrap_angle_rad(theta):
    wrap = (theta + np.pi) % (2 * np.pi) - np.pi
    return wrap


### def compute_log_likelihood

功能：使用给定的初始状态概率、转移矩阵和 emission 参数，计算一段连续 trial 序列的对数似然。

输入：`data`、`params`、`transition_matrix`、`initial_probabilities`。

输出：该 trial 序列的 `log_likelihood`。


In [ ]:
def compute_log_likelihood(
    data,
    params,
    transition_matrix,
    initial_probabilities,
):
    """
    Compute the log-likelihood of one continuous trial sequence.

    Parameters
    ----------
    data : dict
        Must contain:
        - x_rad
        - y_rad
        - coherence
        - prior_std

    params : dict
        Must contain:
        - kappaS: dict mapping coherence to sensory kappa
        - kappaP: dict mapping prior_std to prior kappa

    transition_matrix : array, shape (3, 3)
        Rows are previous states and columns are next states.

    initial_probabilities : array, shape (3,)
        Initial probabilities for Sensory, Prior, and Lapse states.

    Returns
    -------
    log_likelihood : float
        Log-likelihood of this one trial sequence.
    """

    x_rad = np.asarray(data["x_rad"], dtype=float)
    y_rad = np.asarray(data["y_rad"], dtype=float)
    coherence = np.asarray(data["coherence"])
    prior_std = np.asarray(data["prior_std"])

    transition_matrix = np.asarray(
        transition_matrix,
        dtype=float,
    )

    initial_probabilities = np.asarray(
        initial_probabilities,
        dtype=float,
    )

    n_trials = len(y_rad)

    if n_trials == 0:
        raise ValueError("The input data contains no trials.")

    if not (
        len(x_rad)
        == len(y_rad)
        == len(coherence)
        == len(prior_std)
    ):
        raise ValueError("All trial-level arrays must have the same length.")

    if transition_matrix.shape != (3, 3):
        raise ValueError("transition_matrix must have shape (3, 3).")

    if initial_probabilities.shape != (3,):
        raise ValueError("initial_probabilities must have shape (3,).")

    transition_matrix = (
        transition_matrix
        / transition_matrix.sum(axis=1, keepdims=True)
    )

    initial_probabilities = (
        initial_probabilities
        / initial_probabilities.sum()
    )

    emission_matrix = np.zeros((n_trials, 3))

    for t in range(n_trials):

        kappaS_t = params["kappaS"][coherence[t]]
        kappaP_t = params["kappaP"][prior_std[t]]

        sensory_likelihood = vonmises.pdf(
            y_rad[t],
            kappaS_t,
            loc=x_rad[t],
        )

        prior_likelihood = vonmises.pdf(
            y_rad[t],
            kappaP_t,
            loc=0.0,
        )

        lapse_likelihood = 1.0 / (2.0 * np.pi)

        emission_matrix[t] = [
            sensory_likelihood,
            prior_likelihood,
            lapse_likelihood,
        ]

    # Trial 1
    alpha = initial_probabilities * emission_matrix[0]

    scale = alpha.sum()

    if scale <= 0 or not np.isfinite(scale):
        return -np.inf

    alpha = alpha / scale
    log_likelihood = np.log(scale)

    # Trial 2 to Trial T
    for t in range(1, n_trials):

        predicted_probabilities = alpha @ transition_matrix

        alpha = (
            predicted_probabilities
            * emission_matrix[t]
        )

        scale = alpha.sum()

        if scale <= 0 or not np.isfinite(scale):
            return -np.inf

        alpha = alpha / scale
        log_likelihood += np.log(scale)

    return float(log_likelihood)

### def prepare_hmm_data

功能：把原始运动方向和反应坐标转换成以 prior mean 为中心的弧度数据，并整理 HMM 所需变量。

输入：运动方向、反应坐标、prior mean、prior std、coherence、subject id 和 run id。

输出：包含 `x_rad`、`y_rad`、`coherence`、`prior_std`、`block_id` 和 `subject_id` 的字典。


In [9]:
def prepare_hmm_data(
        motion_direction,
        estimate_x,
        estimate_y,
        prior_mean,
        prior_std,
        motion_coherence,
        subject_id,
        run_id,
):
    esti_agl = np.atan2(estimate_y, estimate_x)

    motion_rad = np.deg2rad(motion_direction)
    prior_mean_rad = np.deg2rad(prior_mean)

    x_rad = wrap_angle_rad(motion_rad - prior_mean_rad)
    y_rad = wrap_angle_rad(esti_agl - prior_mean_rad)

    block_id = run_id

    data = {
        "x_rad": x_rad,
        "y_rad": y_rad,
        "coherence": motion_coherence,
        "prior_std": prior_std,
        "block_id": block_id,
        "subject_id": subject_id,
    }

    return data


## initialize functions

### def initialize_parameters

功能：给 Soft EM-HMM 提供初始状态概率、初始转移矩阵和各实验条件下的初始 kappa。

输入：无。

输出：参数字典 `params`。


In [ ]:
def initialize_parameters():
    params = {
        "initial_prob": np.array([0.60, 0.35, 0.05]),

        "transition_matrix": np.array([
            [0.80, 0.15, 0.05],
            [0.15, 0.80, 0.05],
            [0.40, 0.40, 0.20],
        ]),

        "kappaS": {
            0.06: 1.5,
            0.12: 4.5,
            0.24: 20.0,
        },

        "kappaP": {
            80: 0.5,
            40: 1.0,
            20: 6.0,
            10: 30.0,
        },
    }

    return params


## Emission function

### def von_mises_density

功能：计算观察角度在指定 von Mises 分布下的概率密度。

输入：观察角度 `theta`、分布中心 `mu` 和集中参数 `kappa`。

输出：von Mises 概率密度。


In [7]:
def von_mises_density(theta, mu, kappa):
    p = np.exp(kappa * np.cos(theta - mu)) / (2 * np.pi * i0(kappa))
    return p


### def get_sensory_kappa

功能：根据当前 trial 的 coherence 取得对应的 sensory kappa。

输入：sensory kappa 字典 `kappaS` 和当前 `coherence`。

输出：当前条件对应的 sensory kappa。


In [ ]:
def get_sensory_kappa(kappaS, coherence):
    return kappaS[coherence]


### def get_prior_kappa

功能：根据当前 trial 的 prior std 取得对应的 prior kappa。

输入：prior kappa 字典 `kappaP` 和当前 `prior_std`。

输出：当前条件对应的 prior kappa。


In [ ]:
def get_prior_kappa(kappaP, prior_std):
    return kappaP[prior_std]


### def compute_trial_emission_likelihoods

功能：计算一个 trial 在 Sensory、Prior 和 Lapse 三个状态下的 emission likelihood。

输入：当前 response `y_t`、stimulus `x_t`、coherence、prior std 和参数字典。

输出：按 `[Sensory, Prior, Lapse]` 排列的三个 likelihood。


In [ ]:
def compute_trial_emission_likelihoods(y_t, x_t, coherence, prior_std, params):
    bS = von_mises_density(
        y_t,
        x_t,
        get_sensory_kappa(params["kappaS"], coherence),
    )
    bP = von_mises_density(
        y_t,
        0.0,
        get_prior_kappa(params["kappaP"], prior_std),
    )
    bL = 1 / (2 * np.pi)

    return np.array([bS, bP, bL])


### def compute_emission_matrix

功能：逐 trial 调用 emission 函数，生成整段数据的 emission matrix。

输入：整理后的 `data` 和参数字典 `params`。

输出：形状为 `n_trials × 3` 的 emission matrix。


In [ ]:
def compute_emission_matrix(data, params):
    x_rad = data["x_rad"]
    y_rad = data["y_rad"]
    coherence = data["coherence"]
    prior_std = data["prior_std"]

    n_trials = len(x_rad)
    emission_matrix = np.zeros((n_trials, 3))

    for t in range(n_trials):
        emission_matrix[t] = compute_trial_emission_likelihoods(
            y_t=y_rad[t],
            x_t=x_rad[t],
            coherence=coherence[t],
            prior_std=prior_std[t],
            params=params,
        )

    return emission_matrix


## Forward-Backward function

### def forward_pass

功能：从前向后结合初始概率、转移矩阵和当前 emission，计算每个 trial 的预测概率、过滤后概率及序列对数似然。

输入：参数字典 `params` 和一段连续 block 数据 `data`。

输出：`predicted_prob`、`filtered_prob`、`emission_likelihood`、`scales` 和 `log_likelihood`。


In [ ]:
def forward_pass(params, data):
    x_rad = data["x_rad"]
    y_rad = data["y_rad"]
    coherence = data["coherence"]
    prior_std = data["prior_std"]

    n = len(x_rad)
    n_states = 3

    A = params["transition_matrix"]
    initial_probs = params["initial_prob"]

    predicted_prob = np.zeros((n, n_states))
    filtered_prob = np.zeros((n, n_states))
    emission_likelihood = np.zeros((n, n_states))
    scales = np.zeros(n)

    log_likelihood = 0.0

    # Prior state probability before observing trial 0
    predicted_prob[0] = initial_probs

    for i in range(n):
        emission_likelihood[i] = compute_trial_emission_likelihoods(
            y_t=y_rad[i],
            x_t=x_rad[i],
            coherence=coherence[i],
            prior_std=prior_std[i],
            params=params,
        )

        filtered_unnormalized = (
            predicted_prob[i]
            * emission_likelihood[i]
        )

        scales[i] = filtered_unnormalized.sum()

        if scales[i] <= 0:
            raise ValueError(
                f"Forward scaling factor is zero at trial {i}."
            )

        filtered_prob[i] = (
            filtered_unnormalized / scales[i]
        )

        log_likelihood += np.log(scales[i])

        if i < n - 1:
            predicted_prob[i + 1] = (
                filtered_prob[i] @ A
            )

    return (
        predicted_prob,
        filtered_prob,
        emission_likelihood,
        scales,
        log_likelihood,
    )


### def backward_pass

功能：从最后一个 trial 向前计算未来数据对当前隐藏状态的支持。

输入：`params`、连续 block 数据、forward scaling factors 和 emission likelihood。

输出：每个 trial、每个状态的 backward probability `beta`。


In [1]:
def backward_pass(params, data, scales, emission_likelihood):
    x_rad = data["x_rad"]

    n = len(x_rad)
    n_states = 3

    beta = np.zeros((n, n_states))
    A = params["transition_matrix"]

    beta[-1] = [1.0, 1.0, 1.0]

    for t in range(n - 2, -1, -1):
        future_support_next = (
            emission_likelihood[t + 1]
            * beta[t + 1]
        )

        beta[t] = A @ future_support_next
        beta[t] /= scales[t + 1]

    return beta

## M-step update

### def update_state_posteriors

功能：结合 forward 过滤概率和 backward probability，计算每个 trial 的状态后验概率 gamma。

输入：`filtered_prob` 和 `beta`。

输出：形状为 `n_trials × 3` 的 `gamma`。


In [ ]:
def update_state_posteriors(filtered_porb, beta):
    gamma_unnormalized = filtered_porb * beta
    gamma = gamma_unnormalized / gamma_unnormalized.sum(
        axis=1,
        keepdims=True
    )
    return gamma

### def compute_transition_posteriors

功能：计算每一对相邻 trial 在九种状态转移下的联合后验概率 xi。

输入：过滤后状态概率、`beta`、emission likelihood 和 transition matrix。

输出：形状为 `(n_trials - 1) × 3 × 3` 的 `xi`。


In [1]:
def compute_transition_posteriors(
        filtered_prob,
        beta,
        emission_likelihood,
        transition_matrix,
):
    next_state_support = (
        emission_likelihood[1:] * beta[1:]
    )

    xi_unnormalized = (
        filtered_prob[:-1, :, np.newaxis]
        * transition_matrix[np.newaxis, :, :]
        * next_state_support[:, np.newaxis, :]
    )

    normalizers = xi_unnormalized.sum(
        axis=(1, 2),
        keepdims=True,
    )

    if np.any(normalizers <= 0):
        raise ValueError(
            "Some transition posteriors cannot be normalized."
        )

    xi = xi_unnormalized / normalizers

    return xi

### def compute_weighted_resultant_length

功能：使用状态后验概率作为权重，计算圆周误差的加权 resultant length。

输入：圆周误差 `error` 和对应权重 `weights`。

输出：范围通常在 0 到 1 之间的 resultant length `R`。


In [2]:
def compute_weighted_resultant_length(error, weights):
    weighted_cos = np.sum(weights * np.cos(error))
    weighted_sin = np.sum(weights * np.sin(error))

    R = np.sqrt(
        weighted_cos**2 + weighted_sin**2
    ) / np.sum(weights)

    return R

### def estimate_kappa_from_resultant

功能：根据 resultant length 的常用分段近似，估计 von Mises 分布的 kappa。

输入：resultant length `R`。

输出：估计得到的 `kappa`。


In [3]:
def estimate_kappa_from_resultant(R):
    R = np.clip(R, 0.0, 0.999999)

    if R < 0.53:
        kappa = 2 * R + R**3 + 5 * R**5 / 6

    elif R < 0.85:
        kappa = -0.4 + 1.39 * R + 0.43 / (1 - R)

    else:
        kappa = 1 / (R**3 - 4 * R**2 + 3 * R)

    return kappa

### def update_sensory_kappas

功能：按 coherence 条件，用 Sensory 状态后验概率加权 sensory error，更新各 sensory kappa。

输入：`data`、状态后验 `gamma` 和当前 `params`。

输出：更新后的 sensory kappa 字典。


In [4]:
def update_sensory_kappas(data, gamma, params):
    x_rad = data["x_rad"]
    y_rad = data["y_rad"]
    coherence = data["motion_coherence"]

    error_S = wrap_angle_rad(y_rad - x_rad)
    weights_S = gamma[:, 0]

    new_kappaS = {}

    for condition in params["kappaS"].keys():
        mask = coherence == condition

        condition_errors = error_S[mask]
        condition_weights = weights_S[mask]

        R = compute_weighted_resultant_length(
            condition_errors,
            condition_weights,
        )

        new_kappaS[condition] = (
            estimate_kappa_from_resultant(R)
        )

    return new_kappaS

### def update_prior_kappas

功能：按 prior std 条件，用 Prior 状态后验概率加权 prior error，更新各 prior kappa。

输入：`data`、状态后验 `gamma` 和当前 `params`。

输出：更新后的 prior kappa 字典。


In [5]:
def update_prior_kappas(data, gamma, params):
    y_rad = data["y_rad"]
    prior_std = data["prior_std"]

    error_P = wrap_angle_rad(y_rad)
    weights_P = gamma[:, 1]

    new_kappaP = {}

    for condition in params["kappaP"].keys():
        mask = prior_std == condition

        condition_errors = error_P[mask]
        condition_weights = weights_P[mask]

        R = compute_weighted_resultant_length(
            condition_errors,
            condition_weights,
        )

        new_kappaP[condition] = (
            estimate_kappa_from_resultant(R)
        )

    return new_kappaP

### def update_emission_parameters

功能：根据 `update_kappa` 设置，统一更新 sensory 和 prior emission 参数。

输入：`data`、`gamma`、当前 `params` 和 `update_kappa`。

输出：更新后的参数字典。


In [6]:
def update_emission_parameters(data, gamma, params, update_kappa=True):
    if update_kappa:
        params["kappaS"] = update_sensory_kappas(
            data,
            gamma,
            params,
        )

        params["kappaP"] = update_prior_kappas(
            data,
            gamma,
            params,
        )

    return params

### def update_transition_matrix

功能：把所有相邻 trial 的 xi 作为 soft transition counts，逐行归一化得到新的 transition matrix。

输入：`xi` 和可选平滑量 `smoothing`。

输出：形状为 `3 × 3` 的新 transition matrix。


In [ ]:
def update_transition_matrix(xi, smoothing=1e-6):
    transition_counts = xi.sum(axis=0)
    transition_counts = transition_counts + smoothing

    transition_matrix = transition_counts / transition_counts.sum(
        axis=1,
        keepdims=True,
    )

    return transition_matrix


### def m_step

功能：完成一次 M-step，使用 xi 更新 transition matrix，并可选地使用 gamma 更新 kappa。

输入：`data`、`gamma`、`xi`、当前 `params` 和 `update_kappa`。

输出：更新后的参数字典 `new_params`。


In [7]:
def m_step(data, gamma, xi, params, update_kappa=True):
    new_params = params.copy()

    new_params["transition_matrix"] = (
        update_transition_matrix(xi)
    )

    if update_kappa:
        new_params["kappaS"] = (
            update_sensory_kappas(
                data,
                gamma,
                params,
            )
        )

        new_params["kappaP"] = (
            update_prior_kappas(
                data,
                gamma,
                params,
            )
        )

    return new_params

# full-training function

### def compute_total_log_likelihood

功能：把所有 block 的 log-likelihood 相加，得到当前 EM 迭代的总对数似然。

输入：每个 block 的 `block_log_likelihoods`。

输出：总对数似然。


In [ ]:
def compute_total_log_likelihood(block_log_likelihoods):
    total_log_likelihood = np.sum(block_log_likelihoods)
    return float(total_log_likelihood)


### def check_convergence

功能：比较相邻两轮总 log-likelihood，判断 Soft EM 是否达到停止条件。

输入：当前和上一轮 log-likelihood，以及阈值 `tol`。

输出：布尔值，达到阈值时为 `True`。


In [ ]:
def check_convergence(
        current_log_likelihood,
        previous_log_likelihood,
        tol=1e-6,
):
    if previous_log_likelihood is None:
        return False

    difference = abs(
        current_log_likelihood - previous_log_likelihood
    )

    return difference < tol


### def fit_soft_em_hmm

功能：按 block 重复执行 emission、forward-backward 和 M-step，直到收敛或达到最大迭代次数。

输入：完整 `data`、可选初始 `params`、`update_kappa`、`max_iter` 和 `tol`。

输出：最终参数、每个 trial 的 gamma、每对相邻 trial 的 xi、log-likelihood 历史和收敛信息。


In [ ]:
def fit_soft_em_hmm(
        data,
        params=None,
        update_kappa=True,
        max_iter=100,
        tol=1e-6,
):
    if params is None:
        params = initialize_parameters()

    current_params = {
        "initial_prob": params["initial_prob"].copy(),
        "transition_matrix": params["transition_matrix"].copy(),
        "kappaS": params["kappaS"].copy(),
        "kappaP": params["kappaP"].copy(),
    }

    block_ids = np.asarray(data["block_id"])
    unique_blocks = pd.unique(block_ids)

    log_likelihood_history = []
    previous_log_likelihood = None
    converged = False

    final_gamma = None
    final_xi = None

    for iteration in range(max_iter):
        gamma_list = []
        xi_list = []
        block_log_likelihoods = []

        combined_data = {
            "x_rad": [],
            "y_rad": [],
            "coherence": [],
            "prior_std": [],
        }

        for block in unique_blocks:
            mask = block_ids == block

            block_data = {
                "x_rad": np.asarray(data["x_rad"])[mask],
                "y_rad": np.asarray(data["y_rad"])[mask],
                "coherence": np.asarray(data["coherence"])[mask],
                "prior_std": np.asarray(data["prior_std"])[mask],
            }

            (
                predicted_prob,
                filtered_prob,
                emission_likelihood,
                scales,
                block_log_likelihood,
            ) = forward_pass(current_params, block_data)

            beta = backward_pass(
                current_params,
                block_data,
                scales,
                emission_likelihood,
            )

            gamma = update_state_posteriors(
                filtered_prob,
                beta,
            )

            xi = compute_transition_posteriors(
                filtered_prob,
                beta,
                emission_likelihood,
                current_params["transition_matrix"],
            )

            gamma_list.append(gamma)
            xi_list.append(xi)
            block_log_likelihoods.append(block_log_likelihood)

            combined_data["x_rad"].append(block_data["x_rad"])
            combined_data["y_rad"].append(block_data["y_rad"])
            combined_data["coherence"].append(block_data["coherence"])
            combined_data["prior_std"].append(block_data["prior_std"])

        gamma_all = np.concatenate(gamma_list, axis=0)
        xi_all = np.concatenate(xi_list, axis=0)

        for key in combined_data:
            combined_data[key] = np.concatenate(
                combined_data[key],
                axis=0,
            )

        total_log_likelihood = compute_total_log_likelihood(
            block_log_likelihoods
        )
        log_likelihood_history.append(total_log_likelihood)

        converged = check_convergence(
            total_log_likelihood,
            previous_log_likelihood,
            tol,
        )

        final_gamma = gamma_all
        final_xi = xi_all

        if converged:
            break

        current_params = m_step(
            combined_data,
            gamma_all,
            xi_all,
            current_params,
            update_kappa=update_kappa,
        )

        previous_log_likelihood = total_log_likelihood

    results = {
        "params": current_params,
        "gamma": final_gamma,
        "xi": final_xi,
        "log_likelihood_history": log_likelihood_history,
        "converged": converged,
        "n_iterations": len(log_likelihood_history),
    }

    return results


# Viterbi-decoding function

### def viterbi_decode_block

功能：使用训练完成的参数，对一个连续 block 解码最可能的完整隐藏状态路径。

输入：一个 block 的 `data` 和训练后的 `params`。

输出：按 trial 排列的状态编号路径，0、1、2 分别对应 Sensory、Prior、Lapse。


In [ ]:
def viterbi_decode_block(data, params):
    emission_matrix = compute_emission_matrix(data, params)

    n_trials = len(data["x_rad"])
    n_states = 3

    log_initial = np.log(params["initial_prob"])
    log_transition = np.log(params["transition_matrix"])
    log_emission = np.log(emission_matrix)

    delta = np.zeros((n_trials, n_states))
    psi = np.zeros((n_trials, n_states), dtype=int)

    delta[0] = log_initial + log_emission[0]

    for t in range(1, n_trials):
        for current_state in range(n_states):
            previous_scores = (
                delta[t - 1]
                + log_transition[:, current_state]
            )

            psi[t, current_state] = np.argmax(previous_scores)
            delta[t, current_state] = (
                previous_scores[psi[t, current_state]]
                + log_emission[t, current_state]
            )

    state_path = np.zeros(n_trials, dtype=int)
    state_path[-1] = np.argmax(delta[-1])

    for t in range(n_trials - 2, -1, -1):
        state_path[t] = psi[t + 1, state_path[t + 1]]

    return state_path


### def viterbi_decode_all_blocks

功能：对完整数据中的每个 block 分别运行 Viterbi，避免跨 block 连接状态路径。

输入：完整 `data` 和训练后的 `params`。

输出：与原 trial 顺序一致的 Viterbi 状态编号数组。


In [ ]:
def viterbi_decode_all_blocks(data, params):
    block_ids = np.asarray(data["block_id"])
    unique_blocks = pd.unique(block_ids)

    all_state_paths = np.zeros(len(block_ids), dtype=int)

    for block in unique_blocks:
        mask = block_ids == block

        block_data = {
            "x_rad": np.asarray(data["x_rad"])[mask],
            "y_rad": np.asarray(data["y_rad"])[mask],
            "coherence": np.asarray(data["coherence"])[mask],
            "prior_std": np.asarray(data["prior_std"])[mask],
        }

        block_state_path = viterbi_decode_block(
            block_data,
            params,
        )

        all_state_paths[mask] = block_state_path

    return all_state_paths
